In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
# from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import time
import os
# from typing import Tuple, Optional, Sequence
from modules import data_loaders, learning
from modules.models import LMNet



In [ ]:
ptb_path = "Data/ptb/"
fnames = ["ptb.char.train.txt", "ptb.char.valid.txt", "ptb.char.test.txt"]

# config = sys.argv[1] if len(sys.argv) > 1 else "LSTM"
# config = 'LCCL'
config = 'LCRL'
file_name = f"Results/weights_{config}"

n_hidden = 256
# n_hidden = 1
seq_len = 35
grad_clip = 10
batch_size = 32
vocab_size = 10000
learning_rate = 0.002
hid_prop = True
num_epochs = 10
save_fq = 50
print_fq = 1
seed = 0



In [8]:
train_data, valid_data, test_data, word2idx, idx2word = data_loaders.load_PTB_word(ptb_path, *fnames)
train_data = data_loaders.PTB_word(train_data, seq_len, batch_size)
valid_data = data_loaders.PTB_word(valid_data, seq_len, batch_size)
test_data = data_loaders.PTB_word(test_data, seq_len, batch_size)

train_loader = data_loaders.PTBWordLoader(train_data, vocab_size)
valid_loader = data_loaders.PTBWordLoader(valid_data, vocab_size)
test_loader = data_loaders.PTBWordLoader(test_data, vocab_size)

VOCAB_SIZE = 10000


In [ ]:
lm_net = LMNet(vocab_size, n_hidden, config, hid_prop, batch_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lm_net.parameters(), lr=learning_rate)

lm_trainer = learning.LMTrainer(lm_net,
                       optimizer,
                       criterion,
                       num_epochs,
                       train_loader,
                       valid_loader,
                       test_loader)

lm_trainer.train()


epoch 0001/10 | batch 0000/830 | CE 9.2359 | total_loss 0.5151 | tokens 1120 
epoch 0001/10 | batch 0100/830 | CE 6.7293 | total_loss -2.4288 | tokens 1120 
epoch 0001/10 | batch 0200/830 | CE 6.6033 | total_loss -2.6234 | tokens 1120 


KeyboardInterrupt: 

In [ ]:

def generate_text(model, seed_words, word2idx, idx2word, vocab_size, max_len=30, device='cpu'):
    model.eval()
    with torch.no_grad():
        # 1️⃣ Превращаем слова в индексы
        idx_seq = [word2idx[w] for w in seed_words]
        x = torch.tensor([idx_seq], dtype=torch.long, device=device)

        # 2️⃣ One-hot кодировка → (1, seq_len, vocab_size)
        x_onehot = nn.functional.one_hot(x, num_classes=vocab_size).float()

        # 3️⃣ Генерация
        for _ in range(max_len):
            # Прогон через модель
            logits = model(x_onehot)

            # Берём логиты последнего шага
            next_logits = logits[:, -1, :]  # (1, vocab_size)
            probs = nn.functional.softmax(next_logits, dim=-1).squeeze(0)

            # Сэмплируем следующее слово
            next_idx = torch.multinomial(probs, num_samples=1).item()

            # Добавляем новое слово в последовательность
            x = torch.cat([x, torch.tensor([[next_idx]], device=device)], dim=1)

            # Пересчитываем one-hot
            x_onehot = nn.functional.one_hot(x, num_classes=vocab_size).float()

        # 4️⃣ Конвертируем индексы обратно в слова
        generated_words = [idx2word[int(i)] for i in x[0].tolist()]
        return " ".join(generated_words)

seed = ["asbestos"]
print(generate_text(lm_net, seed, word2idx, idx2word, vocab_size=len(word2idx), max_len=50))


asbestos composed fitness responsible november kidder wyoming shakespeare el rica tucker rush seagate coopers affiliated affect madrid cohen fiber urgency club ferry subsidy kronor crossland irving creek cheaper nimitz noriega expecting enthusiasts recovery terminate miniscribe carbide seriously loved tremor favorable saved daly adm. backlash likewise audiences fdic hart-scott-rodino years m$ around


profiling

In [185]:
import torch.profiler as profiler


def test_with_loader(model, loader):
    model.train()
    # -------------------
    # профилируем одну эпоху
    # -------------------
    with profiler.profile(
        activities=[profiler.ProfilerActivity.CPU],
        record_shapes=True,
        with_stack=True
    ) as prof:
        for step, (x, y) in enumerate(loader):
            with profiler.record_function("model_inference"):
                logits = model(x)
            if step >= 2:  # ограничимся батчами для теста
                break
    print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=15))


model = LMNet(vocab_size, n_hidden, config, hid_prop, batch_size)

test_with_loader(model, train_loader)


-----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                         Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
              model_inference        10.32%     113.103ms        94.27%        1.033s     344.447ms             3  
                 aten::matmul         0.09%     999.000us        35.47%     388.809ms       3.503ms           111  
                     aten::mm        35.34%     387.407ms        35.35%     387.492ms       3.491ms           111  
                aten::normal_        23.64%     259.191ms        23.64%     259.191ms      21.599ms            12  
                  aten::randn         0.01%     137.900us        18.48%     202.599ms      22.511ms             9  
                    aten::mul         9.26%     101.476ms         9.77% 